# Skin Lesion Boundary Detection Using Canny Edge Detection
**Pipeline:** Original → Grayscale → Gaussian Filter → Canny → Lesion Boundary → Area & Perimeter

Run the cells top to bottom. Only the upload cell needs your input (5 HAM10000 images).

## Setup

In [ ]:
import os, glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

plt.rcParams["figure.dpi"] = 110

## Task 1 — Load the Image
**Get 5 images:** open the Kaggle dataset page, download any 5 `.jpg` files from `HAM10000_images_part_1`
(or download the zip once and pick 5), then upload them below.
Tip: choose lesions that are clearly darker than the skin around them.

In [ ]:
os.makedirs("images", exist_ok=True)
uploaded = files.upload()          # select your 5 images
for name in uploaded:
    os.replace(name, os.path.join("images", name))

paths = sorted(glob.glob("images/*"))[:5]
print(len(paths), "images loaded:", [os.path.basename(p) for p in paths])

*(Optional alternative: pull straight from Kaggle. This downloads the full ~5 GB dataset, so uploading 5 files is faster.)*
```python
# !pip -q install kagglehub
# import kagglehub, glob
# d = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
# paths = sorted(glob.glob(d + "/**/*.jpg", recursive=True))[:5]
```

## Helper functions

In [ ]:
def load_image(path, max_side=600):
    """Read an image (BGR) and resize so the longest side is <= max_side."""
    bgr = cv2.imread(path)
    h, w = bgr.shape[:2]
    s = max_side / max(h, w)
    if s < 1:
        bgr = cv2.resize(bgr, (int(w * s), int(h * s)))
    return bgr

def preprocess(bgr):
    """Task 2: grayscale + Gaussian filter."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 1.5)
    return gray, blur

def find_boundary(edges):
    """Task 5: edges -> morphological closing -> external contours -> pick lesion contour.
    Returns (contour, score, solidity). Score favours large, compact (convex-ish) shapes
    that do not touch the image border."""
    h, w = edges.shape
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=2)
    cnts, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    best, best_score, best_sol = None, 0.0, 0.0
    for c in cnts:
        a = cv2.contourArea(c)
        if a < 0.01 * h * w or a > 0.8 * h * w:
            continue
        x, y, bw, bh = cv2.boundingRect(c)
        m = 3
        touches = x <= m or y <= m or x + bw >= w - m or y + bh >= h - m
        hull = cv2.contourArea(cv2.convexHull(c))
        solidity = a / hull if hull > 0 else 0.0
        score = a * solidity ** 2 * (0.3 if touches else 1.0)
        if score > best_score:
            best, best_score, best_sol = c, score, solidity
    return best, best_score, best_sol

def area_perimeter(contour, shape):
    """Task 6: pixels inside the boundary (filled mask) + contour length."""
    mask = np.zeros(shape, np.uint8)
    cv2.drawContours(mask, [contour], -1, 255, thickness=-1)
    return int(np.count_nonzero(mask)), float(cv2.arcLength(contour, True)), mask

## Tasks 2–6 — Run the pipeline on all 5 images
For each image this will: convert to grayscale, apply a Gaussian filter, run Canny with three threshold pairs
(50–100, 100–200, 150–250), **auto-select** the pair that gives the clearest closed lesion boundary,
draw it on the original, and compute area and perimeter.

In [ ]:
THRESHOLDS = [(50, 100), (100, 200), (150, 250)]
results, store = [], []

for i, p in enumerate(paths, 1):
    bgr = load_image(p)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray, blur = preprocess(bgr)

    # --- Task 3: three Canny settings
    cands = []
    for lo, hi in THRESHOLDS:
        e = cv2.Canny(blur, lo, hi)
        cnt, score, sol = find_boundary(e)
        cands.append(dict(th=(lo, hi), edges=e, cnt=cnt, score=score,
                          sol=sol, n_edge=int(np.count_nonzero(e))))

    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    for a, c in zip(ax, cands):
        a.imshow(c["edges"], cmap="gray"); a.axis("off")
        a.set_title(f"Canny {c['th'][0]}-{c['th'][1]}  (edge px: {c['n_edge']})", fontsize=9)
    fig.suptitle(f"Image {i}: three Canny threshold settings", y=1.02)
    plt.show()

    # --- Task 4: pick the best (clearest closed lesion boundary)
    best = max(cands, key=lambda d: (round(d["score"]), -d["n_edge"]))  # ties -> fewer (cleaner) edges
    print(f"Image {i} selection scores:",
          {f"{c['th'][0]}-{c['th'][1]}": round(c["score"]) for c in cands},
          "->", f"best = {best['th'][0]}-{best['th'][1]}")

    # --- Task 5 + 6: boundary, area, perimeter
    if best["cnt"] is None:
        print("   No valid lesion contour found for this image.")
        area, perim, overlay = 0, 0.0, rgb.copy()
    else:
        area, perim, _ = area_perimeter(best["cnt"], gray.shape)
        overlay = rgb.copy()
        cv2.drawContours(overlay, [best["cnt"]], -1, (0, 255, 0), 2)

    # --- Required visualization row
    titles = ["Original", "Grayscale", "Gaussian Filter",
              f"Canny {best['th'][0]}-{best['th'][1]}", "Lesion Boundary"]
    imgs = [rgb, gray, blur, best["edges"], overlay]
    fig, ax = plt.subplots(1, 5, figsize=(18, 3.6))
    for a, im, t in zip(ax, imgs, titles):
        a.imshow(im, cmap="gray" if im.ndim == 2 else None); a.set_title(t); a.axis("off")
    fig.suptitle(f"Image {i}  |  Area = {area} px   Perimeter = {perim:.1f} px", y=1.04)
    plt.show()

    results.append({"Image": f"Image {i}", "Best Filter": "Gaussian (5x5, sigma=1.5)",
                    "Edge Method": f"Canny ({best['th'][0]}-{best['th'][1]})",
                    "Area (pixels)": area, "Perimeter (pixels)": round(perim, 1)})
    store.append(dict(gray=gray, best_th=best["th"]))

results_df = pd.DataFrame(results)
results_df

## Task 4 — Why this threshold? (write this in your report, adapt to what *your* plots show)
- **50–100 (low):** keeps even faint gradients, so skin texture, hair and tiny noise also appear as edges. The lesion outline gets buried in clutter and the contour merges with background structures.
- **150–250 (high):** removes almost all noise, but low-contrast parts of the lesion border disappear too, leaving broken edges and an incomplete boundary.
- **100–200 (medium):** usually the best trade-off: strong, mostly continuous lesion border with little background noise, so the boundary closes into one clean contour.

Use the printed selection scores and the three edge maps to confirm which one won on each image, and state it in your own words.

## Final Comparison — Sobel vs Canny with 4 filters
Four filters: **Original** (no filter), **Average**, **Gaussian**, **Median**; each followed by **Sobel** (Otsu-thresholded gradient magnitude) or **Canny** (each image's selected thresholds).

There is no ground-truth mask in this dataset, so quality is measured with proxy metrics (average over the 5 images):
- **Noise handling** = 1 − (fraction of edge pixels that belong to tiny isolated fragments < 30 px). Higher is better.
- **Edge quality** = edge continuity (share of edge pixels in the largest connected edge component). Higher is better.
- **Boundary detection** = solidity of the extracted lesion contour (0 if none found). Higher is better.
- **Overall** = mean of the three scores.

In [ ]:
def sobel_edges(img):
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    mag = cv2.normalize(cv2.magnitude(gx, gy), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, e = cv2.threshold(mag, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return e

def edge_metrics(e):
    n, _, stats, _ = cv2.connectedComponentsWithStats((e > 0).astype(np.uint8), connectivity=8)
    sizes = stats[1:, cv2.CC_STAT_AREA]
    if sizes.size == 0:
        return 0.0, 0.0
    total = sizes.sum()
    return 1 - sizes[sizes < 30].sum() / total, sizes.max() / total

filters = {
    "Original": lambda g: g,
    "Average":  lambda g: cv2.blur(g, (5, 5)),
    "Gaussian": lambda g: cv2.GaussianBlur(g, (5, 5), 1.5),
    "Median":   lambda g: cv2.medianBlur(g, 5),
}

rows = []
for fname, f in filters.items():
    for ename in ["Sobel", "Canny"]:
        noise_s, qual_s, bound_s = [], [], []
        for s in store:
            fi = f(s["gray"])
            e = sobel_edges(fi) if ename == "Sobel" else cv2.Canny(fi, *s["best_th"])
            ns, q = edge_metrics(e)
            _, _, sol = find_boundary(e)
            noise_s.append(ns); qual_s.append(q); bound_s.append(sol)
        rows.append({"Method": f"{fname} + {ename}",
                     "Noise Handling": np.mean(noise_s),
                     "Edge Quality": np.mean(qual_s),
                     "Boundary Detection": np.mean(bound_s)})

comp = pd.DataFrame(rows)
comp["Overall Performance"] = comp[["Noise Handling", "Edge Quality", "Boundary Detection"]].mean(axis=1)

# numeric table
display(comp.round(3))

# rating table (Poor / Fair / Good by ranking terciles) for the report
rating = comp[["Method"]].copy()
for col in ["Noise Handling", "Edge Quality", "Boundary Detection", "Overall Performance"]:
    rating[col] = pd.qcut(comp[col].rank(method="first"), 3, labels=["Poor", "Fair", "Good"])
rating

## Conclusion (adapt to your results)
- Gaussian + Canny normally ranks at or near the top: the Gaussian filter smooths noise while keeping edges reasonably sharp, and Canny's non-maximum suppression and hysteresis give thin, connected edges.
- Sobel produces thick, noisy edges with no hysteresis, so it handles noise worse, especially with no pre-filter.
- Average filtering blurs the lesion border; Median filtering removes hair/speckle noise well while preserving edges, so it is a strong second choice.
- Limitation: edge-based boundaries struggle when the lesion has low contrast, fuzzy borders or heavy hair.